In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
import os
import sys

with open('/data/hyz/RAT/dataset/text_files/blip_laion_cc_sbu_558k.json', 'r') as f:
    pretrain = json.load(f)
# 指定设备为卡6
device = torch.device("cuda:6" if torch.cuda.is_available() else "cpu")

hf_path = '/data/hyz/RAT/checkpoints/tinyllava-phi'
model = AutoModelForCausalLM.from_pretrained(hf_path, trust_remote_code=True)
model.to(device)
config = model.config
tokenizer = AutoTokenizer.from_pretrained(hf_path, use_fast=False, model_max_length = config.tokenizer_model_max_length,padding_side = config.tokenizer_padding_side)


/data/hyz/.conda/envs/tinyllava/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:04<00:00,  2.15s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [2]:
from PIL import Image
image = '/data/hyz/RAT/dataset/llava/llava_pretrain/images/'+pretrain[0]['image']
prompt = ' the caption of th image is: '
caption = pretrain[0]['conversations'][1]['value']

image_processor = model.vision_tower._image_processor

def expand2square(pil_img, background_color):
    width, height = pil_img.size
    if width == height:
        return pil_img
    elif width > height:
        result = Image.new(pil_img.mode, (width, width), background_color)
        result.paste(pil_img, (0, (width - height) // 2))
        return result
    else:
        result = Image.new(pil_img.mode, (height, height), background_color)
        result.paste(pil_img, ((height - width) // 2, 0))
        return result
    
def process_images(images, image_processor, model_cfg):
    image_aspect_ratio = getattr(model_cfg, "image_aspect_ratio", None)
    new_images = []
    if image_aspect_ratio == 'pad':
        for image in images:
            image = expand2square(image, tuple(int(x*255) for x in image_processor.image_mean))
            image = image_processor.preprocess(image, return_tensors='pt')['pixel_values'][0]
            new_images.append(image)
    else:
        return image_processor(images, return_tensors='pt')['pixel_values']
    if all(x.shape == new_images[0].shape for x in new_images):
        new_images = torch.stack(new_images, dim=0)
    return new_images

if image is not None:
    prompt = "<image>" + '\n' + prompt 

def tokenizer_image_token(prompt, tokenizer, image_token_index=-200, return_tensors=None):
    prompt_chunks = [tokenizer(chunk).input_ids for chunk in prompt.split('<image>')]

    def insert_separator(X, sep):
        return [ele for sublist in zip(X, [sep]*len(X)) for ele in sublist][:-1]

    input_ids = []
    offset = 0
    if len(prompt_chunks) > 0 and len(prompt_chunks[0]) > 0 and prompt_chunks[0][0] == tokenizer.bos_token_id:
        offset = 1
        input_ids.append(prompt_chunks[0][0])

    for x in insert_separator(prompt_chunks, [image_token_index] * (offset + 1)):
        input_ids.extend(x[offset:])

    if return_tensors is not None:
        if return_tensors == 'pt':
            return torch.tensor(input_ids, dtype=torch.long)
        raise ValueError(f'Unsupported tensor type: {return_tensors}')
    return input_ids

In [3]:
image = Image.open(image).convert("RGB")
image_tensor = process_images(image, image_processor, model.config).to(model.device)
input_ids = (
    tokenizer_image_token(prompt, tokenizer, -200, return_tensors="pt")
    .unsqueeze(0).to(model.device)
)

In [4]:
image_tensor.shape, input_ids.shape, prompt, caption

(torch.Size([1, 3, 384, 384]),
 torch.Size([1, 10]),
 '<image>\n the caption of th image is: ',
 'select luxury furniture 3 - inch gel memory foam mattress topper')

In [5]:
image_features = model.encode_images(image_tensor).squeeze(0).to(device)

In [6]:
image_features.shape

torch.Size([728, 2560])

In [ ]:
import torch.nn as nn
import torch
from transformers.models.perceiver.modeling_perceiver import PerceiverTextPreprocessor
from transformers.models.perceiver.modeling_perceiver import PerceiverImagePreprocessor
from transformers.models.perceiver.modeling_perceiver import PerceiverMultimodalPreprocessor
from transformers.models.perceiver.modeling_perceiver import PerceiverEncoder
from transformers.models.perceiver.modeling_perceiver import PerceiverEmbeddings
from transformers.models.perceiver.modeling_perceiver import PerceiverConfig
from transformers import PerceiverTokenizer
class PerceiverMultiModalEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        
        # 创建图像和文本的预处理器
        image_preprocessor = PerceiverImagePreprocessor(
            config,
            prep_type="conv1x1",  # 使用1x1卷积处理图像
            spatial_downsample=1,
            out_channels=256,
            position_encoding_type="trainable",
            concat_or_add_pos="concat",
            project_pos_dim=256,
            trainable_position_encoding_kwargs=dict(
                num_channels=256,
                index_dims=config.image_size**2,
            ),
        )
        
        text_preprocessor = PerceiverTextPreprocessor(config)
        
        # 创建多模态预处理器
        self.preprocessor = PerceiverMultimodalPreprocessor(
            modalities={
                "image": image_preprocessor,
                "text": text_preprocessor
            },
            min_padding_size=4
        )
        
        # 创建Perceiver编码器
        self.embeddings = PerceiverEmbeddings(config)
        self.encoder = PerceiverEncoder(
            config,
            kv_dim=self.preprocessor.num_channels
        )
        
    def forward(self, inputs):
        """
        Args:
            inputs: 字典,包含:
                - image: shape (batch_size, channels, height, width)
                - text: shape (batch_size, sequence_length)
                
        Returns:
            latent: shape (batch_size, num_latents, d_latents)
        """
        # 1. 预处理输入
        processed_inputs, _, _ = self.preprocessor(inputs)
        
        # 2. 获取latent的初始值
        batch_size = processed_inputs.shape[0]
        latent = self.embeddings(batch_size)
        print(latent.shape)
        # 3. 通过编码器获取最终的latent表示
        encoder_outputs = self.encoder(
            latent,
            inputs=processed_inputs,
        )
        
        # 返回最后一层的latent表示
        return encoder_outputs[0]  # shape: (batch_size, num_latents, d_latents)

# 使用示例:
# config = PerceiverConfig.from_pretrained("/data/hyz/vlm/multimodal-perceiver")
perceiver_tokenizer = PerceiverTokenizer.from_pretrained("/data/hyz/vlm/perceiver")

encoding = perceiver_tokenizer(caption)
encoding = torch.tensor(encoding['input_ids']).unsqueeze(dim=0).to(model.device)

config = PerceiverConfig(
    num_latents=256,          # latent序列的长度
    d_latents=512,           # latent的维度
    d_model=256,             # 预处理后的输入维度
    num_self_attends_per_block=8,  # self-attention层数
    num_blocks=1,            # 编码器block数量
    num_self_attention_heads=8,    # self-attention的头数
    num_cross_attention_heads=8,   # cross-attention的头数
    qk_channels=512,         # 添加这个参数，确保能被8整除
    v_channels=512, 
    image_size=384,          # 输入图像大小
    vocab_size=30522,        # 词表大小
    max_position_embeddings=512,  # 最大文本长度
)
# 2. 创建模型
perceiver_model = PerceiverMultiModalEncoder(config)
perceiver_model = perceiver_model.to(model.device)
# # 3. 准备输入数据
# inputs = {
#     "image": torch.randn(2, 3, 224, 224),  # 批量大小为2的图像
#     "text": torch.randint(0, config.vocab_size, (2, 128))  # 批量大小为2的文本
# }
# encoding_image_tensor = torch.randn(1,3,224,224).to(model.device)


In [ ]:
inputs = {
    'image': image_tensor,
    'text' : encoding
}
latent = perceiver_model(inputs)
print("Latent shape:", latent.shape)  

In [ ]:
inputs['image'].shape, inputs['text'].shape

In [ ]:
latent